In [0]:
# Load the required Silver columns
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orig = spark.table("workspace.mortgage_risk.silver_origination").select(
    "loan_id",
    "first_payment_date",
    "fico",
    "ltv",
    "cltv",
    "dti",
    "original_upb",
    "original_rate",
    "original_term",
    "mi_pct",
    "borrowers",
    "units",
    "purpose",
    "occupancy",
    "property_type",
    "state",
    "channel",
    "source_vintage"
)

perf = spark.table("workspace.mortgage_risk.silver_performance").select(
    "loan_id",
    "period",
    "current_upb",
    "current_rate",
    "estimated_ltv",
    "loan_age",
    "remaining_term",
    "delinquency_months",
    "delinquency_unknown",
    "serious_delinquency_observed",
    "zero_balance_code"
)

In [0]:
# Define calendar-based history windows
# Each calendar month gets a consecutive number.
history = perf.withColumn(
    "month_index",
    F.year("period") * 12 + F.month("period")
)

loan_window = Window.partitionBy("loan_id").orderBy("month_index")

last_3_months = loan_window.rangeBetween(-2, 0)
last_6_months = loan_window.rangeBetween(-5, 0)

all_past_months = loan_window.rangeBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

history = history.withColumn(
    "late_payment",
    F.when(F.col("delinquency_months") >= 1, 1).otherwise(0)
)

history = history.withColumn(
    "exit_record",
    F.when(F.col("zero_balance_code").isNotNull(), 1).otherwise(0)
)

In [0]:
# Calculate payment-history features and coverage
history = history.withColumn(
    "history_months_6m",
    F.count("*").over(last_6_months)
)

history = history.withColumn(
    "unknown_months_6m",
    F.sum("delinquency_unknown").over(last_6_months)
)

history = history.withColumn(
    "late_months_3m",
    F.sum("late_payment").over(last_3_months)
)

history = history.withColumn(
    "late_months_6m",
    F.sum("late_payment").over(last_6_months)
)

history = history.withColumn(
    "max_delinquency_3m",
    F.max("delinquency_months").over(last_3_months)
)

history = history.withColumn(
    "max_delinquency_6m",
    F.max("delinquency_months").over(last_6_months)
)

# Check for serious events observed through the prediction month.
history = history.withColumn(
    "serious_event_to_date",
    F.max("serious_delinquency_observed").over(all_past_months)
)

history = history.withColumn(
    "exit_to_date",
    F.max("exit_record").over(all_past_months)
)

In [0]:
# Select eligible predictions and join origination
eligible = history.filter(
    (F.col("history_months_6m") == 6)
    & (F.col("unknown_months_6m") == 0)
    & F.col("delinquency_months").between(0, 2)
    & (F.col("current_upb") > 0)
    & (F.col("serious_event_to_date") == 0)
    & (F.col("exit_to_date") == 0)
)

gold_features = eligible.join(orig, on="loan_id", how="inner")

gold_features = gold_features.filter(
    F.col("period") >= F.add_months(F.col("first_payment_date"), 5)
)

gold_features = gold_features.withColumnRenamed(
    "period", "prediction_month"
)

# Guard against a missing or invalid denominator.
gold_features = gold_features.withColumn(
    "balance_ratio",
    F.when(
        F.col("original_upb") > 0,
        F.col("current_upb").cast("double")
        / F.col("original_upb").cast("double")
    )
)

gold_features = gold_features.withColumn(
    "additional_lien_ltv",
    F.col("cltv") - F.col("ltv")
)

# Remove temporary calculation columns.
gold_features = gold_features.drop(
    "month_index",
    "late_payment",
    "exit_record",
    "zero_balance_code",
    "serious_delinquency_observed",
    "serious_event_to_date",
    "exit_to_date"
)


In [0]:
# output 
display(
    gold_features.select(
        "loan_id",
        "prediction_month",
        "fico",
        "delinquency_months",
        "history_months_6m",
        "unknown_months_6m",
        "late_months_3m",
        "late_months_6m",
        "max_delinquency_6m",
        "balance_ratio"
    ).limit(20)
)

display(
    gold_features.groupBy(F.year("prediction_month").alias("year"))
    .count()
    .orderBy("year")
)

loan_id,prediction_month,fico,delinquency_months,history_months_6m,unknown_months_6m,late_months_3m,late_months_6m,max_delinquency_6m,balance_ratio
F16Q10000144,2016-08-01,729,0,6,0,0,0,0,0.98989898989899
F16Q10000144,2016-09-01,729,0,6,0,0,0,0,0.9875423232323233
F16Q10000144,2016-10-01,729,0,6,0,0,0,0,0.9860320707070708
F16Q10000144,2016-11-01,729,0,6,0,0,0,0,0.9845169696969697
F16Q10000144,2016-12-01,729,0,6,0,0,0,0,0.9829969696969697
F16Q10000144,2017-01-01,729,0,6,0,0,0,0,0.9814721212121212
F16Q10000144,2017-02-01,729,0,6,0,0,0,0,0.9799423737373737
F16Q10000144,2017-03-01,729,0,6,0,0,0,0,0.9784077272727273
F16Q10000144,2017-04-01,729,0,6,0,0,0,0,0.9768681313131314
F16Q10000144,2017-05-01,729,0,6,0,0,0,0,0.9753236363636363


year,count
2016,55988
2017,533966
2018,1052770
2019,1497634
2020,1589182
2021,1468454
2022,1717458
2023,2168976
2024,2602185
2025,2947069


In [0]:
# Refresh the Gold feature table.
assert gold_features.limit(1).count() > 0, "No eligible feature rows found."

(
    gold_features.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.mortgage_risk.gold_mortgage_features")
)

print("Gold features table refreshed.")


Gold features table refreshed.


In [0]:
# check the number of prediction rows by year
features = spark.table(
    "workspace.mortgage_risk.gold_mortgage_features"
)

yearly_counts = (
    features
    .withColumn("prediction_year", F.year("prediction_month"))
    .groupBy("prediction_year")
    .agg(
        F.count("*").alias("prediction_rows"),
        F.countDistinct("loan_id").alias("unique_loans")
    )
    .orderBy("prediction_year")
)

display(yearly_counts)

prediction_year,prediction_rows,unique_loans
2016,55988,19415
2017,533966,67505
2018,1052770,112937
2019,1497634,154799
2020,1589182,179957
2021,1468454,167154
2022,1717458,173649
2023,2168976,207755
2024,2602185,245695
2025,2947069,277473


In [0]:
# Load features and the full performance history
features = spark.table(
    "workspace.mortgage_risk.gold_mortgage_features"
)

perf = spark.table(
    "workspace.mortgage_risk.silver_performance"
).select(
    "loan_id",
    "period",
    "serious_delinquency_observed",
    "delinquency_unknown",
    "zero_balance_code",
    "zero_balance_date"
)

perf = perf.withColumn(
    "month_index",
    F.year("period") * 12 + F.month("period")
)

loan_window = Window.partitionBy("loan_id").orderBy("month_index")

# Start next month and end twelve months after the prediction month
next_12_months = loan_window.rangeBetween(1, 12)

In [0]:
# Examine each future outcome window
outcomes = perf.withColumn(
    "future_records",
    F.count("*").over(next_12_months)
)

outcomes = outcomes.withColumn(
    "future_unknown_months",
    F.coalesce(
        F.sum("delinquency_unknown").over(next_12_months),
        F.lit(0)
    )
)

outcomes = outcomes.withColumn(
    "future_event_month",
    F.min(
        F.when(
            F.col("serious_delinquency_observed") == 1,
            F.col("period")
        )
    ).over(next_12_months)
)

# Use the earlier of the exit's reporting and effective months
exit_month = F.when(
    F.col("zero_balance_code").isNotNull(),
    F.least(
        F.col("period"),
        F.coalesce(F.col("zero_balance_date"), F.col("period"))
    )
)

outcomes = outcomes.withColumn(
    "future_exit_month",
    F.min(exit_month).over(next_12_months)
)

outcomes = outcomes.select(
    "loan_id",
    F.col("period").alias("prediction_month"),
    "future_records",
    "future_unknown_months",
    "future_event_month",
    "future_exit_month"
)

In [0]:
# Assign the initial labels
gold_draft = features.join(
    outcomes,
    on=["loan_id", "prediction_month"],
    how="left"
)

event_observed = (
    F.col("future_event_month").isNotNull()
    & (
        F.col("future_exit_month").isNull()
        | (F.col("future_event_month") <= F.col("future_exit_month"))
    )
)

complete_without_event = (
    (F.col("future_records") == 12)
    & (F.col("future_unknown_months") == 0)
    & F.col("future_event_month").isNull()
    & F.col("future_exit_month").isNull()
)

gold_draft = gold_draft.withColumn(
    "label_12m",
    F.when(event_observed, 1)
     .when(complete_without_event, 0)
     .otherwise(F.lit(None).cast("int"))
)

gold_draft = gold_draft.withColumn(
    "label_reason",
    F.when(event_observed, "serious_event_observed")
     .when(complete_without_event, "complete_without_event")
     .when(
         F.col("future_exit_month").isNotNull(),
         "exit_needs_review"
     )
     .when(
         F.col("future_unknown_months") > 0,
         "unknown_future_status"
     )
     .otherwise("insufficient_followup_or_gap")
)

# Earliest reporting month that establishes this outcome.
gold_draft = gold_draft.withColumn(
    "label_available_month",
    F.when(event_observed, F.col("future_event_month"))
     .when(
         complete_without_event,
         F.add_months(F.col("prediction_month"), 12)
     )
)

In [0]:
# Check the label distribution
label_summary = (
    gold_draft
    .groupBy("label_12m", "label_reason")
    .count()
    .orderBy("label_reason")
)

display(label_summary)

label_12m,label_reason,count
0,complete_without_event,11406847
null,exit_needs_review,1980901
null,insufficient_followup_or_gap,2859463
1,serious_event_observed,159890


In [0]:
# Next resolve verified voluntary payoffs. target ends when a loan leaves the portfolio: a verified payoff without an earlier serious event can receive 0

In [0]:
# Find loans whose first recorded exit is a verified payoff
perf = spark.table(
    "workspace.mortgage_risk.silver_performance"
).select(
    "loan_id",
    "period",
    "current_upb",
    "zero_balance_code",
    "zero_balance_date",
    "delinquency_unknown",
    "serious_delinquency_observed"
)

exit_records = perf.filter(
    F.col("zero_balance_code").isNotNull()
)

# Keep the first exit and its corresponding details together.
first_exits = exit_records.groupBy("loan_id").agg(
    F.min(
        F.struct(
            F.col("period").alias("report_month"),
            F.col("zero_balance_code").alias("code"),
            F.col("zero_balance_date").alias("effective_month"),
            F.col("current_upb").alias("balance")
        )
    ).alias("first_exit")
)

# Require code 01, consistent dates, and zero remaining balance.
payoffs = first_exits.filter(
    (F.col("first_exit.code") == "01")
    & (
        F.col("first_exit.report_month")
        == F.col("first_exit.effective_month")
    )
    & (F.col("first_exit.balance") == 0)
).select(
    "loan_id",
    F.col("first_exit.report_month").alias("payoff_month")
)

In [0]:
# Select unresolved predictions with payoff inside their horizon
payoff_candidates = gold_draft.filter(
    F.col("label_12m").isNull()
    & (F.col("label_reason") == "exit_needs_review")
).select(
    "loan_id",
    "prediction_month",
    "future_exit_month"
)

payoff_candidates = payoff_candidates.join(
    payoffs,
    on="loan_id",
    how="inner"
)

payoff_candidates = payoff_candidates.filter(
    (F.col("payoff_month") > F.col("prediction_month"))
    & (
        F.col("payoff_month")
        <= F.add_months(F.col("prediction_month"), 12)
    )
    & (F.col("payoff_month") == F.col("future_exit_month"))
).drop("future_exit_month")

In [0]:
# Check every month from prediction through payoff
candidates = payoff_candidates.alias("c")
payments = perf.alias("p")

matched_months = candidates.join(
    payments,
    (F.col("c.loan_id") == F.col("p.loan_id"))
    & (F.col("p.period") > F.col("c.prediction_month"))
    & (F.col("p.period") <= F.col("c.payoff_month")),
    how="left"
)

matched_months = matched_months.select(
    F.col("c.loan_id").alias("loan_id"),
    F.col("c.prediction_month").alias("prediction_month"),
    F.col("c.payoff_month").alias("payoff_month"),
    F.col("p.period").alias("observed_month"),
    F.col("p.delinquency_unknown").alias("unknown_status"),
    F.col("p.serious_delinquency_observed").alias("serious_event")
)

payoff_check = matched_months.groupBy(
    "loan_id", "prediction_month", "payoff_month"
).agg(
    F.count("observed_month").alias("observed_months"),
    F.sum("unknown_status").alias("unknown_months"),
    F.sum("serious_event").alias("serious_events")
)

payoff_check = payoff_check.withColumn(
    "expected_months",
    F.months_between("payoff_month", "prediction_month").cast("int")
)

payoff_check = payoff_check.withColumn(
    "coverage_complete",
    (F.col("observed_months") == F.col("expected_months"))
    & (F.col("unknown_months") == 0)
    & (F.col("serious_events") == 0)
)

verified_payoffs = payoff_check.filter(
    F.col("coverage_complete") == True
).select(
    "loan_id",
    "prediction_month",
    F.col("payoff_month").alias("verified_payoff_month")
)

In [0]:
#vUpdate only the verified payoff labels
gold_final = gold_draft.join(
    verified_payoffs,
    on=["loan_id", "prediction_month"],
    how="left"
)

gold_final = gold_final.withColumn(
    "payoff_verified",
    F.col("label_12m").isNull()
    & F.col("verified_payoff_month").isNotNull()
)

gold_final = gold_final.withColumn(
    "label_reason",
    F.when(
        F.col("payoff_verified"),
        "paid_off_without_event"
    ).otherwise(F.col("label_reason"))
)

gold_final = gold_final.withColumn(
    "label_available_month",
    F.when(
        F.col("payoff_verified"),
        F.col("verified_payoff_month")
    ).otherwise(F.col("label_available_month"))
)

gold_final = gold_final.withColumn(
    "label_12m",
    F.when(F.col("payoff_verified"), 0)
     .otherwise(F.col("label_12m"))
)

gold_final = gold_final.drop(
    "payoff_verified",
    "verified_payoff_month"
)

display(
    gold_final.groupBy("label_12m", "label_reason")
    .count()
    .orderBy("label_reason")
)

label_12m,label_reason,count
0,complete_without_event,11406847
null,exit_needs_review,12914
null,insufficient_followup_or_gap,2859463
0,paid_off_without_event,1967987
1,serious_event_observed,159890


In [0]:
# Check the final Gold data before saving it.
from pyspark.sql import functions as F

known_label = F.col("label_12m").isNotNull()

invalid_label = (
    known_label
    & ~F.col("label_12m").isin(0, 1)
)

invalid_label_date = (
    known_label
    & (
        F.col("label_available_month").isNull()
        | (
            F.col("label_available_month")
            <= F.col("prediction_month")
        )
        | (
            F.col("label_available_month")
            > F.add_months(F.col("prediction_month"), 12)
        )
    )
)

missing_key = (
    F.col("loan_id").isNull()
    | (F.trim(F.col("loan_id")) == "")
    | F.col("prediction_month").isNull()
)

gold_check = gold_final.agg(
    F.count("*").alias("total_rows"),
    F.count("label_12m").alias("labelled_rows"),
    F.sum(
        F.when(F.col("label_12m") == 1, 1).otherwise(0)
    ).alias("positive_rows"),
    F.sum(
        F.when(missing_key, 1).otherwise(0)
    ).alias("missing_keys"),
    F.sum(
        F.when(invalid_label, 1).otherwise(0)
    ).alias("invalid_labels"),
    F.sum(
        F.when(invalid_label_date, 1).otherwise(0)
    ).alias("invalid_label_dates")
)

result = gold_check.first().asDict()
print(result)

# Counts can change when new source data arrives.
assert result["total_rows"] > 0, "The final Gold dataset is empty."
assert result["total_rows"] == features.count(), (
    "Label construction changed the number of feature rows."
)
assert (
    0 <= result["positive_rows"]
    <= result["labelled_rows"]
    <= result["total_rows"]
), "The label counts are inconsistent."

assert result["missing_keys"] == 0, "Missing loan ID or prediction month."
assert result["invalid_labels"] == 0, "Unexpected label values."
assert result["invalid_label_dates"] == 0, "Invalid outcome availability dates."


{'total_rows': 16407101, 'labelled_rows': 13534724, 'positive_rows': 159890, 'missing_keys': 0, 'invalid_labels': 0, 'invalid_label_dates': 0}


In [0]:
duplicate_predictions = (
    gold_final.groupBy("loan_id", "prediction_month")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_predictions.limit(1).count() == 0, \
    "Duplicate loan-month predictions found."

print("Gold checks passed.")

Gold checks passed.


In [0]:
# Save only after the checks in cells 17 and 18 pass.
(
    gold_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.mortgage_risk.gold_mortgage_12m")
)

print("Final Gold table refreshed successfully.")


Final Gold table refreshed successfully.
